This simulation uses Elo ratings from <https://eloratings.net> to measure team strength and update it after each simulated game. The Elo implementation is based on FiveThirtyEight’s NFL forecasting game (<https://github.com/morales-felix/nfl-elo-game>).

Notes on Elo implementation:  

- Per <https://eloratings.net/about>, the K constant is set to 60 as this is a World Cup competition.  
- Probabilities given by the Elo rating system are binary. I came up with a workaround to convert them to ternary probabilities given that association football (a.k.a. soccer) admits three outcomes after a match (win, tie, lose).  
- I did not simulate scorelines. Rather, I simply used probabilities to decide whether a team would win, tie, or lose. As such, I did not use the goal difference multiplier specified in <https://eloratings.net/about>  
- I'll be happy to talk about the workaround, but I wouldn't take it as gospel. There might be ways to do this, but I did not research it. Wanted to have fun, not produce an academic-paper-worthy method, nor a sellable product.  

In [ ]:
import pandas as pd
import csv
from tqdm import tqdm

from src.world_cup_simulator import *

Since I want to simulate the group stage many times to generate a distribution of outcomes, I will use the `joblib` module to parallelize the simulation. This will allow me to run the simulation many times in a reasonable amount of time. That requires me to use a function to simulate the group stage and return the results.

In [ ]:
def run_group_stage_simulation(n, j):
    """
    Run a simulation of the group stage of the World Cup
    """
    
    teams_pd = pd.read_csv("data/roster.csv")
    
    for i in range(n):
        games = read_games("data/matches.csv")
        teams = {}
    
        for row in [
            item for item in csv.DictReader(open("data/roster.csv"))
            ]:
            teams[row['team']] = {
                'name': row['team'],
                'rating': float(row['rating']),
                'points': 0
                }
    
        simulate_group_stage(
            games,
            teams,
            ternary=True
            )
    
        collector = []
        for key in teams.keys():
            collector.append(
                {"team": key,
                 f"pts_simulation{i+1}": teams[key]['points'],
                 f"rating_simulation{i+1}": teams[key]['rating']}
            )

        temp = pd.DataFrame(collector)
        teams_pd = pd.merge(teams_pd, temp)
    
    pts_sim_cols = [
        a for a in teams_pd.columns if a.startswith("pts_simulation")]
    rating_sim_cols = [
        a for a in teams_pd.columns if a.startswith("rating_simulation")]
    teams_pd[
        f"avg_pts_{j+1}"
        ] = teams_pd[pts_sim_cols].mean(axis=1)
    teams_pd[
        f"avg_rating_{j+1}"
        ] = teams_pd[rating_sim_cols].mean(axis=1)
    not_sim = [
        b for b in teams_pd.columns if "simulation" not in b]
    simulation_result = teams_pd[not_sim]
    
    return simulation_result

### Simulate group stage

#### The gist is to read from two files: One defining the match schedule, the other with teams and their relative strengths (given by Elo ratings prior to the start of the event)

In [ ]:
# Reads in the matches and teams as dictionaries and proceeds with that data type
n = 1000 # How many simulations to run
m = 1000 # How many simulation results to collect
from joblib import Parallel, delayed

roster_pd = Parallel(n_jobs=-1)(
    delayed(run_group_stage_simulation)(
        n, j) for j in tqdm(range(m)))

for t in tqdm(range(m)):
    if t == 0:
        roster = pd.merge(
            roster_pd[t],
            roster_pd[t+1]
            )
    elif t >= 2:
        roster = pd.merge(
            roster,
            roster_pd[t]
            )
    else:
        pass

In [ ]:
point_sim_cols = [i for i in roster.columns if i.startswith("avg_pts_")]
rating_sim_cols = [i for i in roster.columns if i.startswith("avg_rating_")]

In [ ]:
roster['avg_sim_pts'] = roster[point_sim_cols].mean(axis=1)
roster['99%CI_low'] = roster[point_sim_cols] \
    .quantile(q=0.005, axis=1)
roster['99%CI_high'] = roster[point_sim_cols] \
    .quantile(q=0.995, axis=1)

roster['avg_sim_rating'] = roster[rating_sim_cols] \
    .mean(axis=1) \
    .round() \
    .astype(int)
roster['99%CI_rating_low'] = roster[rating_sim_cols] \
    .quantile(q=0.005, axis=1) \
    .round() \
    .astype(int)
roster['99%CI_rating_high'] = roster[rating_sim_cols] \
    .quantile(q=0.995, axis=1) \
    .round() \
    .astype(int)

In [ ]:
not_sim = [
    j for j in roster.columns
    if not j.startswith("avg_pts_")
    and not j.startswith("avg_rating_")
    ]

#### Simulation is done, now take a look at the results for the group stage

In [ ]:
group_stage_ranked = roster[not_sim].copy()
group_stage_ranked = group_stage_ranked.sort_values(
    by=["group", "avg_sim_pts"],
    ascending=[True, False]
    )
group_stage_ranked

### Which teams are likely to qualify for the knockout stage?

In [ ]:
group_stage_result_base = group_stage_ranked.copy()
group_stage_result_base = group_stage_result_base.sort_values(
    by=["group", "avg_sim_pts"],
    ascending=[True, False]
    )
group_stage_result_base["group_rank"] = (
    group_stage_result_base
    .groupby("group")
    .cumcount()
    .add(1)
    )

top_two_teams = group_stage_result_base[
    group_stage_result_base["group_rank"] <= 2
    ]
best_third_place_teams = (
    group_stage_result_base[group_stage_result_base["group_rank"] == 3]
    .sort_values("avg_sim_pts", ascending=False)
    .head(8)
    )

group_stage_result = (
    pd.concat([top_two_teams, best_third_place_teams], ignore_index=True)
    [["group", "team", "group_rank"]]
    .sort_values(["group", "group_rank"])
    .reset_index(drop=True)
    )

group_stage_result.to_csv("data/group_stage_result.csv", index=False)
group_stage_result

#### Build out the playoff bracket

In [ ]:
updated_ratings = group_stage_result_base[
    ["team", "avg_sim_rating"]
    ].rename(columns={"avg_sim_rating": "rating"})
playoff_roster = (
    group_stage_result[["team"]]
    .merge(updated_ratings, on="team", how="left")
    )

if playoff_roster["rating"].isna().any():
    missing = playoff_roster.loc[playoff_roster["rating"].isna(), "team"].tolist()
    raise ValueError(f"Missing Elo ratings for: {missing}")

playoff_roster.to_csv("data/playoff_roster.csv", index=False)
playoff_roster

In [ ]:
import re
import subprocess
from pathlib import Path

def third_place_matchups_from_annex_c(pdf_path="FWC26_regulations_EN.pdf"):
    try:
        completed = subprocess.run(
            ["pdftotext", "-layout", str(Path(pdf_path)), "-"],
            check=True,
            capture_output=True
            )
    except FileNotFoundError as exc:
        raise RuntimeError(
            "pdftotext is required to parse Annex C from FWC26_regulations_EN.pdf."
            ) from exc

    text = completed.stdout.decode("utf-8", errors="replace")
    row_pattern = re.compile(
        r"^\s*(\d{1,3})\s+" + r"\s+".join([r"(3[A-L])"] * 8) + r"\s*$"
        )
    assignment_targets = ["1A", "1B", "1D", "1E", "1G", "1I", "1K", "1L"]
    matchups = {}

    for line in text.splitlines():
        match = row_pattern.match(line)
        if not match:
            continue

        option = int(match.group(1))
        assignments = match.groups()[1:]
        third_groups_key = "".join(sorted(slot[1] for slot in assignments))
        matchups[third_groups_key] = {
            "option": option,
            "matchups": dict(zip(assignment_targets, assignments))
            }

    if len(matchups) != 495:
        raise ValueError(f"Expected 495 Annex C matchup combinations, found {len(matchups)}")

    return matchups

third_place_groups = group_stage_result.loc[
    group_stage_result["group_rank"] == 3,
    "group"
    ].tolist()
selection_key = "".join(sorted(third_place_groups))
selected_assignment = third_place_matchups_from_annex_c().get(selection_key)

if selected_assignment is None:
    raise ValueError(f"No third-place assignment found for groups: {selection_key}")

third_place_option = selected_assignment["option"]
third_place_opponents = selected_assignment["matchups"]
print(f"Using Annex C option {third_place_option} for third-place groups {selection_key}")
group_stage_lookup = group_stage_result.set_index(["group", "group_rank"])["team"]

def slot_to_team(slot):
    return group_stage_lookup.loc[(slot[1], int(slot[0]))]

match_rows = []

def add_match(match, home_slot, away_slot, to_match, stage):
    match_rows.append({
        "match": match,
        "home_team": "" if home_slot == "" else slot_to_team(home_slot),
        "away_team": "" if away_slot == "" else slot_to_team(away_slot),
        "elo_prob_home": "",
        "result_home": "",
        "advances": "",
        "to_match": to_match,
        "loses": "",
        "penalties": "",
        "stage": stage
        })

round_of_32 = [
    ("2A", "2B", 16),
    ("1E", third_place_opponents["1E"], 17),
    ("1F", "2C", 16),
    ("1C", "2F", 18),
    ("1I", third_place_opponents["1I"], 17),
    ("2E", "2I", 18),
    ("1A", third_place_opponents["1A"], 19),
    ("1L", third_place_opponents["1L"], 19),
    ("1D", third_place_opponents["1D"], 21),
    ("1G", third_place_opponents["1G"], 21),
    ("2K", "2L", 20),
    ("1H", "2J", 20),
    ("1B", third_place_opponents["1B"], 23),
    ("1J", "2H", 22),
    ("1K", third_place_opponents["1K"], 23),
    ("2D", "2G", 22)
    ]

for match, (home_slot, away_slot, to_match) in enumerate(round_of_32):
    add_match(match, home_slot, away_slot, to_match, "round_of_32")

future_matches = [
    (16, 24, "round_of_16"),
    (17, 24, "round_of_16"),
    (18, 26, "round_of_16"),
    (19, 26, "round_of_16"),
    (20, 25, "round_of_16"),
    (21, 25, "round_of_16"),
    (22, 27, "round_of_16"),
    (23, 27, "round_of_16"),
    (24, 28, "quarterfinals"),
    (25, 28, "quarterfinals"),
    (26, 29, "quarterfinals"),
    (27, 29, "quarterfinals"),
    (28, 30, "semifinals"),
    (29, 30, "semifinals"),
    (30, 31, "final"),
    (31, 31, "third_place")
    ]

for match, to_match, stage in future_matches:
    add_match(match, "", "", to_match, stage)

playoff_matches = pd.DataFrame(
    match_rows,
    columns=[
        "match",
        "home_team",
        "away_team",
        "elo_prob_home",
        "result_home",
        "advances",
        "to_match",
        "loses",
        "penalties",
        "stage"
        ]
    )

playoff_matches.to_csv("data/playoff_matches.csv", index=False)
playoff_matches

### Simulating knockout stage  
Here's where it gets interesting

In [ ]:
# Now, doing the Monte Carlo simulations
n = 10000
playoff_results_teams = []
playoff_results_stage = []

for i in tqdm(range(n)):
    overall_result_teams = dict()
    overall_result_stage = dict()
    games = read_games("data/playoff_matches.csv")
    teams = {}
    
    for row in [
        item for item in csv.DictReader(open("data/playoff_roster.csv"))]:
        teams[row['team']] = {
            'name': row['team'],
            'rating': float(row['rating'])
            }
    
    simulate_playoffs(games, teams, ternary=True)
    
    playoff_pd = pd.DataFrame(games)
    
    # This is for collecting results of simulations per team
    for key in teams.keys():
        overall_result_teams[key] = collect_playoff_results(
            key,
            playoff_pd
            )
    playoff_results_teams.append(overall_result_teams)
    
    # Now, collecting results from stage-perspective
    overall_result_stage['whole_bracket'] = playoff_pd['advances'].to_list()
    overall_result_stage['Round_of_16'] = playoff_pd.loc[playoff_pd['stage'] == 'round_of_32', 'advances'].to_list()
    overall_result_stage['Quarterfinals'] = playoff_pd.loc[playoff_pd['stage'] == 'round_of_16', 'advances'].to_list()
    overall_result_stage['Semifinals'] = playoff_pd.loc[playoff_pd['stage'] == 'quarterfinals', 'advances'].to_list()
    overall_result_stage['Final'] = playoff_pd.loc[playoff_pd['stage'] == 'semifinals', 'advances'].to_list()
    overall_result_stage['third_place_match'] = playoff_pd.loc[playoff_pd['stage'] == 'semifinals', 'loses'].to_list()
    overall_result_stage['fourth_place'] = playoff_pd.loc[playoff_pd['stage'] == 'third_place', 'loses'].to_list()[0]
    overall_result_stage['third_place'] = playoff_pd.loc[playoff_pd['stage'] == 'third_place', 'advances'].to_list()[0]
    overall_result_stage['second_place'] = playoff_pd.loc[playoff_pd['stage'] == 'final', 'loses'].to_list()[0]
    overall_result_stage['Champion'] = playoff_pd.loc[playoff_pd['stage'] == 'final', 'advances'].to_list()[0]

    for match_number in range(16, 32):
        overall_result_stage[f'match{match_number}'] = list(
            playoff_pd.loc[match_number, ['home_team', 'away_team']]
            )
    
    playoff_results_stage.append(overall_result_stage)

In [ ]:
results_teams = pd.DataFrame(playoff_results_teams)

In [ ]:
results_teams['Spain'].value_counts()

In [ ]:
results_stage = pd.DataFrame(playoff_results_stage)

In [ ]:
results_stage['Final'].value_counts()